# WRN-28-4 + SE Attention — Animals-5 (Midterm)

**Dataset:** Custom Animals (5 classes: butterfly, cat, chicken, dog, horse)  
**Model:** WideResNet-28-4 with SE Attention (M1)  
**Requirement:** Train ~100 epochs at 224×224 and 32×32

**Kaggle Input:**
- `animals-5-custom-train` → `/kaggle/input/animals-5-custom-train/train/`
- `animals-5-custom-test` → `/kaggle/input/animals-5-custom-test/test/`

## 1. Config

⚠️ **Chạy 2 lần:** lần 1 `IMG_SIZE = 224`, lần 2 đổi thành `IMG_SIZE = 32` rồi **Restart & Run All**.

In [ ]:
import os, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# =============================================
# ĐỔI GIÁ TRỊ NÀY RỒI RESTART & RUN ALL
IMG_SIZE = 224    # Lần 1: 224 — Lần 2: 32
# =============================================

NUM_CLASSES = 5
BATCH_SIZE = 8 if IMG_SIZE == 224 else 128
EPOCHS = 100
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
DROPOUT = 0.3
LABEL_SMOOTHING = 0.1
DEPTH = 16
WIDEN_FACTOR = 2

# === PATHS ===
TRAIN_DIR = '/kaggle/input/datasets/nyantony/animals-5-custom-train/train'
TEST_DIR = '/kaggle/input/datasets/nyantony/animals-5-custom-test/test'
OUT_DIR = f'/kaggle/working/animals_wrn_{IMG_SIZE}'
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Image size: {IMG_SIZE}x{IMG_SIZE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify
for label, path in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    classes = sorted(os.listdir(path))
    counts = {c: len(os.listdir(os.path.join(path, c))) for c in classes}
    print(f'{label}: {counts}')

## 2. Model — WideResNet-28-4 + SE

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        w = F.adaptive_avg_pool2d(x, 1).view(b, c)
        w = F.relu(self.fc1(w), inplace=True)
        w = torch.sigmoid(self.fc2(w)).view(b, c, 1, 1)
        return x * w


class WideResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.se = SEBlock(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride, bias=False)

    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x), inplace=True))
        out = self.dropout(out)
        out = self.conv2(F.relu(self.bn2(out), inplace=True))
        out = self.se(out)
        return out + self.shortcut(x)


class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=4, dropout=0.3, num_classes=5):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        ch = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]

        self.conv1 = nn.Conv2d(3, ch[0], 3, 1, 1, bias=False)
        self.group1 = self._make_group(ch[0], ch[1], n, stride=1, dropout=dropout)
        self.group2 = self._make_group(ch[1], ch[2], n, stride=2, dropout=dropout)
        self.group3 = self._make_group(ch[2], ch[3], n, stride=2, dropout=dropout)
        self.bn_final = nn.BatchNorm2d(ch[3])
        self.fc = nn.Linear(ch[3], num_classes)
        self._init_weights()

    def _make_group(self, in_ch, out_ch, n_blocks, stride, dropout):
        layers = [WideResBlock(in_ch, out_ch, stride, dropout)]
        for _ in range(1, n_blocks):
            layers.append(WideResBlock(out_ch, out_ch, 1, dropout))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.group1(x)
        x = self.group2(x)
        x = self.group3(x)
        x = F.relu(self.bn_final(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1)
        x = torch.flatten(x, 1)
        return self.fc(x)

## 3. Data Loading & Augmentation

**Resize xảy ra ở đây** — `transforms.Resize()`. Dataset gốc giữ nguyên.

In [ ]:
import numpy as np


class Cutout:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        h, w = img.shape[1], img.shape[2]
        y = np.random.randint(h)
        x = np.random.randint(w)
        y1, y2 = max(0, y - self.size // 2), min(h, y + self.size // 2)
        x1, x2 = max(0, x - self.size // 2), min(w, x + self.size // 2)
        img[:, y1:y2, x1:x2] = 0.0
        return img


# ImageNet mean/std (phù hợp cho ảnh thực tế)
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

cutout_size = IMG_SIZE // 4       # 224->56, 32->8
crop_padding = IMG_SIZE // 8      # 224->28, 32->4

train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),          # ← RESIZE
    T.RandomCrop(IMG_SIZE, padding=crop_padding),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    Cutout(cutout_size),
])

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),          # ← RESIZE
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

trainset = ImageFolder(TRAIN_DIR, transform=train_transform)
testset = ImageFolder(TEST_DIR, transform=test_transform)

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Classes: {trainset.classes}')
print(f'Train: {len(trainset)} | Test: {len(testset)}')
print(f'Image size: {IMG_SIZE}x{IMG_SIZE}')

## 4. Init Model

In [ ]:
model = WideResNet(depth=DEPTH, widen_factor=WIDEN_FACTOR,
                    dropout=DROPOUT, num_classes=NUM_CLASSES).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {num_params:,}')

## 5. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                      weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

CKPT_PATH = os.path.join(OUT_DIR, 'last_checkpoint.pth')
start_epoch = 0
best_acc = 0.0
history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_acc = ckpt['best_acc']
    history = ckpt['history']
    print(f'▶ Resumed from epoch {start_epoch}, best_acc={best_acc:.2f}%')
else:
    print('▶ Training from scratch.')


def train_one_epoch(loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * targets.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, 100.0 * correct / total


print(f'Starting training — epochs {start_epoch+1} to {EPOCHS}')
t0 = time.time()

for epoch in range(start_epoch, EPOCHS):
    lr_now = optimizer.param_groups[0]['lr']
    train_loss, train_acc = train_one_epoch(train_loader)
    val_loss, val_acc = evaluate(test_loader)
    scheduler.step()

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(round(train_loss, 5))
    history['train_acc'].append(round(train_acc, 2))
    history['val_loss'].append(round(val_loss, 5))
    history['val_acc'].append(round(val_acc, 2))
    history['lr'].append(round(lr_now, 6))

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_model.pth'))

    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_acc': best_acc,
        'history': history,
    }, CKPT_PATH)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        elapsed = time.time() - t0
        print(f'Epoch {epoch+1:>3d}/{EPOCHS} | '
              f'train_loss={train_loss:.4f} train_acc={train_acc:.1f}% | '
              f'val_loss={val_loss:.4f} val_acc={val_acc:.1f}% | '
              f'best={best_acc:.2f}% | lr={lr_now:.6f} | {elapsed/60:.1f}min')

with open(os.path.join(OUT_DIR, 'training_log.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f'\n✔ Done! Best test accuracy: {best_acc:.2f}%')
print(f'  Total time: {(time.time()-t0)/60:.1f} min')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['epoch'], history['train_acc'], label='Train Acc')
axes[0].plot(history['epoch'], history['val_acc'], label='Test Acc')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title(f'Accuracy — WRN-{DEPTH}-{WIDEN_FACTOR} on Animals ({IMG_SIZE}x{IMG_SIZE})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['epoch'], history['train_loss'], label='Train Loss')
axes[1].plot(history['epoch'], history['val_loss'], label='Test Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title(f'Loss — WRN-{DEPTH}-{WIDEN_FACTOR} on Animals ({IMG_SIZE}x{IMG_SIZE})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'charts_{IMG_SIZE}.png'), dpi=150)
plt.show()

print(f'Best test accuracy ({IMG_SIZE}x{IMG_SIZE}): {best_acc:.2f}%')

## 7. Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

summary_text = f"""============================================
MODEL SUMMARY — WideResNet-{DEPTH}-{WIDEN_FACTOR} + SE Attention
============================================
Dataset:          Animals-5 (Custom)
Image size:       {IMG_SIZE}x{IMG_SIZE}
Classes:          {NUM_CLASSES} ({', '.join(trainset.classes)})
Train samples:    {len(trainset)}
Test samples:     {len(testset)}
Total params:     {total_params:,}
Trainable params: {trainable_params:,}
Epochs:           {EPOCHS}
Batch size:       {BATCH_SIZE}
Optimizer:        SGD (lr={LR}, momentum={MOMENTUM}, wd={WEIGHT_DECAY}, nesterov)
Scheduler:        CosineAnnealingLR
Augmentation:     Resize + RandomCrop + HFlip + ColorJitter + Cutout
Label smoothing:  {LABEL_SMOOTHING}
Dropout:          {DROPOUT}
Best test acc:    {best_acc:.2f}%
============================================
"""
print(summary_text)

with open(os.path.join(OUT_DIR, 'model_summary.txt'), 'w') as f:
    f.write(summary_text)
    f.write('\n--- Epoch-by-epoch log ---\n')
    for i in range(len(history['epoch'])):
        f.write(f"Epoch {history['epoch'][i]:>3d} | "
                f"train_loss={history['train_loss'][i]:.5f} "
                f"train_acc={history['train_acc'][i]:.2f}% | "
                f"val_loss={history['val_loss'][i]:.5f} "
                f"val_acc={history['val_acc'][i]:.2f}%\n")

print(f'\nOutput files:')
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')